# 02 ? PDF ingestion

Turn a cached filing into inspectable pages before chunking. Run from the project environment after fetching data; extraction uses no network or model APIs.

```mermaid
flowchart LR
    A[SEC PDF] --> B[Source hash and page indexes]
    B --> C[Page text and extraction diagnostics]
    C --> D[Later chunking]
    C --> E[Evidence location audit]
    F[FinanceBench annotations] --> E
```

In [ ]:
import json
import os
from pathlib import Path
from time import perf_counter

from finsight.financebench import audit_evidence_locations, load_dataset
from finsight.financebench.paths import pdf_path
from finsight.ingestion import extract_pdf

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").is_file():
    ROOT = ROOT.parent
DATA_DIR = Path(os.environ.get("FINSIGHT_DATA_DIR", ROOT / "data/financebench"))
COMPANY = "AMD"
dataset = load_dataset(DATA_DIR, company=COMPANY)
dataset.referenced_doc_names

## Page provenance

A hash identifies the exact PDF bytes; parser version and extraction mode describe how text was produced. Physical page indexes start at zero. A viewer's page position is index + 1; printed page labels can differ.

In [ ]:
parsed_by_doc = {}
extraction_summary = []
for name in dataset.referenced_doc_names:
    started = perf_counter()
    parsed = extract_pdf(pdf_path(DATA_DIR, name))
    parsed_by_doc[name] = parsed
    extraction_summary.append(
        {
            "document": name,
            "sha256": parsed.source_sha256,
            "parser": f"{parsed.parser_name} {parsed.parser_version}",
            "mode": parsed.extraction_mode,
            "pages": parsed.page_count,
            "seconds": round(perf_counter() - started, 3),
            "diagnostics": parsed.diagnostics,
        }
    )
extraction_summary

## Text versus layout

PDF text extraction recovers characters and an inferred reading order. Layout mode adds spacing to approximate visual alignment; neither mode guarantees structured tables. Compare the same financial page in both modes and inspect numbers, parentheses, headers and columns.

In [ ]:
COMPARISON_DOC = max(dataset.documents, key=lambda doc: doc.doc_period).doc_name
plain = parsed_by_doc[COMPARISON_DOC]
annotated_indexes = {
    evidence.evidence_page_num
    for question in dataset.questions
    for evidence in question.evidence
    if evidence.doc_name == COMPARISON_DOC
}
financial_pages = [
    page
    for page in plain.pages
    if page.page_index in annotated_indexes
    and "consolidated statements of cash flows" in page.text.casefold()
]
fallback = next((page for page in plain.pages if page.status == "text"), None)
COMPARISON_PAGE_INDEX = (
    financial_pages[0].page_index if financial_pages else (fallback.page_index if fallback else 0)
)
layout = extract_pdf(pdf_path(DATA_DIR, COMPARISON_DOC), mode="layout")
print(f"{COMPARISON_DOC}: index {COMPARISON_PAGE_INDEX}")
for label, document in (("plain", plain), ("layout", layout)):
    page = document.pages[COMPARISON_PAGE_INDEX] if document.pages else None
    print()
    print(label.upper())
    print(page.text[:1400] if page else "No physical pages")

## Pages without text

Keep empty and failed pages so later citations retain their positions. No extracted text can mean a blank page, an image or unsupported content; inspect the PDF before deciding whether OCR is needed.

In [ ]:
page_diagnostics = [
    {
        "document": name,
        "index": page.page_index,
        "status": page.status,
        "diagnostics": page.diagnostics,
    }
    for name, document in parsed_by_doc.items()
    for page in document.pages
    if page.status != "text"
]
page_diagnostics

## Evidence alignment

Gold annotations identify known support locations. This audit checks source identity, bounds and text presence, not retrieval quality or answer correctness. Compare an annotated excerpt with its parsed page; different wording alone is not an extraction failure.

Explore another company by fetching its filings explicitly and changing `COMPANY`. The generic extractor also accepts any local text PDF.

In [ ]:
report = audit_evidence_locations(dataset, parsed_by_doc)
location_rows = [
    {
        "question_id": row.question_id,
        "annotation": row.evidence_index,
        "document": row.doc_name,
        "index": row.page_index,
        "viewer_position": row.page_index + 1,
        "source_verified": row.source_verified,
        "in_bounds": row.in_bounds,
        "status": row.page_status,
    }
    for row in report.locations
]
location_rows

In [ ]:
example = next((row for row in report.locations if row.source_verified and row.in_bounds), None)
if example:
    question = next(q for q in dataset.questions if q.financebench_id == example.question_id)
    evidence = question.evidence[example.evidence_index]
    page = parsed_by_doc[example.doc_name].pages[example.page_index]
    print("ANNOTATED SUPPORT", evidence.evidence_text[:400])
    print()
    print("PARSED PAGE PREVIEW", page.text[:1000])
for finding in report.findings:
    print(f"{finding.severity.upper()} {finding.code}: {finding.message}")

In [ ]:
summary = {
    "company": dataset.company,
    "documents": len(parsed_by_doc),
    "pages": sum(doc.page_count for doc in parsed_by_doc.values()),
    "no_text_pages": sum(p.status == "no_text" for d in parsed_by_doc.values() for p in d.pages),
    "error_pages": sum(p.status == "error" for d in parsed_by_doc.values() for p in d.pages),
    "evidence_annotations": report.annotation_count,
    "unique_evidence_locations": report.unique_location_count,
    "text_annotations": report.text_annotation_count,
    "location_audit_ok": report.ok,
}
print(json.dumps(summary, indent=2))